# Sesión 02 – Fundamentos de Spark

**Tema:** DataFrames, lazy evaluation, execution plan y procesamiento textual con RDD.

**Dataset:** `biblia_ntv_.csv` en `/opt/data/`.

**Propósito práctico:** Explorar datos con DataFrames, comprender la diferencia entre transformaciones y acciones, y aplicar procesamiento textual básico con RDD.

## 1. Crear la SparkSession

Para esta práctica trabajaremos en modo local dentro del contenedor Docker. Se usa `local[*]` para aprovechar los cores disponibles y una cantidad moderada de particiones de shuffle para evitar sobrecarga en un dataset pequeño.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 00:30:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Cargar el dataset como DataFrame

El archivo está montado desde la carpeta `data/` del proyecto hacia `/opt/data/` dentro del contenedor.

In [2]:
df = spark.read.csv(
    "/opt/data/biblia_ntv_.csv",
    header=True,
    inferSchema=True
)

df

DataFrame[_c0: int, libro: string, capitulo: int, verso: int, texto: string]

## 3. Explorar estructura y primeros registros

Aquí observamos el esquema y verificamos que el dataset contiene columnas como índice, libro, capítulo, verso y texto.

In [3]:
df.printSchema()
df.show(5, truncate=False)

root
 |-- _c0: integer (nullable = true)
 |-- libro: string (nullable = true)
 |-- capitulo: integer (nullable = true)
 |-- verso: integer (nullable = true)
 |-- texto: string (nullable = true)

+---+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_c0|libro  |capitulo|verso|texto                                                                                                                                                            |
+---+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0  |Génesis|1       |1    |1 El relato de la creación En el principio, Dios creó los cielos y la tierra.                                                                                    |
|1  |Génesis|1       |2    |2 La tierra n

26/03/30 00:33:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , libro, capitulo, verso, texto
 Schema: _c0, libro, capitulo, verso, texto
Expected: _c0 but found: 
CSV file: file:///opt/data/biblia_ntv_.csv


## 4. Exploración con DataFrames

Aplicamos operaciones de selección y filtrado. Estas operaciones son **transformaciones**, por lo que Spark todavía no ejecuta todo inmediatamente.

In [4]:
df.select("libro", "capitulo", "verso").show(10, truncate=False)

df.filter(df.texto.contains("Dios")).select("libro", "capitulo", "verso", "texto").show(5, truncate=False)

+-------+--------+-----+
|libro  |capitulo|verso|
+-------+--------+-----+
|Génesis|1       |1    |
|Génesis|1       |2    |
|Génesis|1       |3    |
|Génesis|1       |4    |
|Génesis|1       |5    |
|Génesis|1       |6    |
|Génesis|1       |7    |
|Génesis|1       |8    |
|Génesis|1       |9    |
|Génesis|1       |10   |
+-------+--------+-----+
only showing top 10 rows
+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|libro  |capitulo|verso|texto                                                                                                                                                            |
+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Génesis|1       |1    |1 El relato de la creación En el princip

## 5. Lazy evaluation y acciones

Construimos un flujo de transformaciones. La ejecución real ocurre recién cuando llamamos a una **acción** como `show()` o `count()`.

In [5]:
df_transformado = (
    df.select("libro", "capitulo", "verso", "texto")
      .filter(df.texto.contains("amor") | df.texto.contains("fe") | df.texto.contains("Dios"))
)

# Hasta aquí Spark solo construye el plan
df_transformado

DataFrame[libro: string, capitulo: int, verso: int, texto: string]

In [6]:
# Acción: aquí recién Spark ejecuta
df_transformado.show(10, truncate=False)

# Otra acción
df_transformado.count()

+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|libro  |capitulo|verso|texto                                                                                                                                                            |
+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Génesis|1       |1    |1 El relato de la creación En el principio, Dios creó los cielos y la tierra.                                                                                    |
|Génesis|1       |2    |2 La tierra no tenía forma y estaba vacía, y la oscuridad cubría las aguas profundas; y el Espíritu de Dios se movía en el aire sobre la superficie de las aguas.|
|Génesis|1       |3    |3 Entonces Dios dijo: «Que haya luz»; y h

6615

## 6. Analizar el plan de ejecución

Usamos `explain()` para observar cómo Spark organiza el procesamiento. Es una base importante para comprender rendimiento y optimización.

In [7]:
df_transformado.explain(True)

== Parsed Logical Plan ==
'Filter 'or('or('contains(texto#21, amor), 'contains(texto#21, fe)), 'contains(texto#21, Dios))
+- Project [libro#18, capitulo#19, verso#20, texto#21]
   +- Relation [_c0#17,libro#18,capitulo#19,verso#20,texto#21] csv

== Analyzed Logical Plan ==
libro: string, capitulo: int, verso: int, texto: string
Filter ((Contains(texto#21, amor) OR Contains(texto#21, fe)) OR Contains(texto#21, Dios))
+- Project [libro#18, capitulo#19, verso#20, texto#21]
   +- Relation [_c0#17,libro#18,capitulo#19,verso#20,texto#21] csv

== Optimized Logical Plan ==
Project [libro#18, capitulo#19, verso#20, texto#21]
+- Filter ((Contains(texto#21, amor) OR Contains(texto#21, fe)) OR Contains(texto#21, Dios))
   +- Relation [_c0#17,libro#18,capitulo#19,verso#20,texto#21] csv

== Physical Plan ==
*(1) Filter ((Contains(texto#21, amor) OR Contains(texto#21, fe)) OR Contains(texto#21, Dios))
+- FileScan csv [libro#18,capitulo#19,verso#20,texto#21] Batched: false, DataFilters: [((Contains(tex

## 7. Paso de DataFrame a RDD

Ahora tomamos solo la columna `texto` y la convertimos a RDD para trabajar la lógica tipo MapReduce.

In [8]:
rdd = df.select("texto").rdd.map(lambda x: x.texto)
rdd.take(5)

['1 El relato de la creación En el principio, Dios creó los cielos y la tierra.',
 '2 La tierra no tenía forma y estaba vacía, y la oscuridad cubría las aguas profundas; y el Espíritu de Dios se movía en el aire sobre la superficie de las aguas.',
 '3 Entonces Dios dijo: «Que haya luz»; y hubo luz.',
 '4 Y Dios vio que la luz era buena. Luego separó la luz de la oscuridad.',
 '5 Dios llamó a la luz «día» y a la oscuridad «noche». Y pasó la tarde y llegó la mañana, así se cumplió el primer día.']

## 8. Filtrar versículos por palabra clave

Buscamos versículos que contengan la palabra **amor**.

In [9]:
textos_amor = rdd.filter(lambda linea: "amor" in linea.lower())
textos_amor.take(5)

['16 los jebuseos, los amorreos, los gergeseos,',
 '7 Luego dieron la vuelta y llegaron a En-mispat (que ahora se llama Cades) y conquistaron todo el territorio de los amalecitas y también a los amorreos que vivían en Hazezon-tamar.',
 '13 Uno de los hombres de Lot escapó y le contó todo a Abram, el hebreo, que vivía cerca del robledo que pertenecía a Mamre, el amorreo. Mamre y sus parientes, Escol y Aner, eran aliados de Abram.',
 '16 Cuando hayan pasado cuatro generaciones, tus descendientes regresarán aquí, a esta tierra, porque los pecados de los amorreos no ameritan aún su destrucción».',
 '21 los amorreos, los cananeos, los gergeseos y los jebuseos».']

## 9. Aplicar flatMap + map + reduceByKey

Se implementa un conteo distribuido de palabras. Antes de contar, realizamos una limpieza básica para reducir ruido por signos de puntuación.

In [10]:
import re
from operator import add

palabras = rdd.flatMap(
    lambda linea: re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()).split()
)

pares = palabras.filter(lambda p: p != "").map(lambda palabra: (palabra, 1))
conteo = pares.reduceByKey(add)

conteo.takeOrdered(20, key=lambda x: -x[1])

[('de', 40372),
 ('y', 27162),
 ('a', 22768),
 ('que', 22566),
 ('el', 22490),
 ('la', 19948),
 ('los', 17711),
 ('en', 13823),
 ('se', 8717),
 ('señor', 8006),
 ('del', 7890),
 ('no', 7153),
 ('con', 6946),
 ('su', 6912),
 ('las', 6699),
 ('por', 6699),
 ('para', 6459),
 ('lo', 6199),
 ('al', 5911),
 ('un', 5617)]

## 10. Reto de práctica

Resolver los siguientes puntos:
1. Obtener las 10 palabras más frecuentes.
2. Determinar cuántas veces aparece la palabra `dios`.
3. Extraer 5 versículos donde aparezca `fe`.

In [11]:
top10 = conteo.takeOrdered(10, key=lambda x: -x[1])
top10

[('de', 40372),
 ('y', 27162),
 ('a', 22768),
 ('que', 22566),
 ('el', 22490),
 ('la', 19948),
 ('los', 17711),
 ('en', 13823),
 ('se', 8717),
 ('señor', 8006)]

In [12]:
conteo.filter(lambda x: x[0] == "dios").collect()

[('dios', 5038)]

In [13]:
rdd.filter(lambda x: "fe" in x.lower()).take(5)

['22 Entonces Dios los bendijo con las siguientes palabras: «Sean fructíferos y multiplíquense. Que los peces llenen los mares y las aves se multipliquen sobre la tierra».',
 '28 Luego Dios los bendijo con las siguientes palabras: «Sean fructíferos y multiplíquense. Llenen la tierra y gobiernen sobre ella. Reinen sobre los peces del mar, las aves del cielo y todos los animales que corren por el suelo».',
 '32 Cuando Noé tenía quinientos años, fue padre de Sem, Cam y Jafet.',
 '10 Noé fue padre de tres hijos: Sem, Cam y Jafet.',
 '16 Deja una abertura de cuarenta y seis centímetros por debajo del techo, alrededor de toda la barca. Pon la puerta en uno de los costados y construye tres pisos dentro de la barca: inferior, medio y superior.']

## 11. Cierre breve

**Preguntas para reflexión:**
- ¿Qué parte resolviste con DataFrame y cuál con RDD?
- ¿En qué momento Spark ejecutó realmente el trabajo?
- ¿Qué observaste en `explain()` sobre el plan lógico y físico?

**Entregable sugerido:** notebook con resultados visibles e interpretación breve.